# P3-v2 read-only post-primary analysis

Run on CPU only after **both** frozen backbones have completed 18/18 full units. This notebook downloads no checkpoint and performs no model inference. It verifies the private arrays and manifests against the frozen source/construct replay, computes all twelve descriptive cells and source-ID-clustered intervals, and writes a private aggregate report plus a source-ID-free CSV. It never modifies P1/P2/P3-v1 or the 36 prediction units. Lower-link SQL remains explicitly unavailable because lower-link quantiles were not archived; it is not imputed from medians.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, subprocess, sys

REPO = Path('/content/tsfm-covariate-faithfulness')
REPO_URL = 'https://github.com/FlyMe2star/tsfm-covariate-faithfulness.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements/p3-preflight.txt')], cwd=REPO, check=True)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('Analysis code commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
from covfaith_p3_v2_analysis import analyze_full_archives
from covfaith_p3.data import sha256_file

DATA_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p3_v2_source_data')
ARCHIVE_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p3_v2_model_v1')
report = analyze_full_archives(REPO, DATA_ROOT, ARCHIVE_ROOT)
report_path = ARCHIVE_ROOT / 'analysis_v1/p3_v2_analysis_report.json'
matrix_path = ARCHIVE_ROOT / 'analysis_v1/complete_twelve_cell_matrix.csv'
assert report['counts']['complete_cell_count'] == 12
assert report['counts']['valid_scenarios_per_backbone'] == 571
print('Result status:', report['result_status'])
print('Report SHA-256:', sha256_file(report_path))
print('Matrix SHA-256:', sha256_file(matrix_path))
print(json.dumps(report['decision'], indent=2, ensure_ascii=False))
print('All twelve cells:')
print(matrix_path.read_text(encoding='utf-8'))
print('Private report:', report_path)
